# Traffic Violation Detection - v7
*(v4 architecture + v6 OCR/helmet improvements + v7 plate-staleness and crop-by-vehicle-type fixes)*

| ID | Area | Change |
|----|------|--------|
| V6-O1 | OCR | Interval 50->25 (balance frequency vs GPU load) |
| V6-O2 | OCR | Dual-crop: primary bottom-25% + fallback bottom-40% (car/default only as of v7; see V7-O1) |
| V6-O3 | OCR | Target 64px height (was 120//h, uncapped) |
| V6-O4 | OCR | CLAHE on plate L-channel before threshold |
| V6-O5 | OCR | paragraph=False, min_size=8 for character-level results |
| V6-O6 | OCR | Length-weighted confidence; floor per fragment |
| V6-O7 | OCR | Dual-polarity: run on both binary and inverted; pick better |
| V6-O8 | OCR | Temporal voting: min 2 valid reads before committing |
| V6-O9 | OCR | Remove double-gate only for avg_conf (format gate kept) |
| **V7-O1** | **OCR** | **Crop band selected by vehicle class (motorcycle/bus/truck/car/bicycle) instead of one fixed band for all vehicles** |
| V6-H1 | Helmet | Person-bbox head extraction with IoU matching |
| V6-H2 | Helmet | Head crop: top 30% of matched person bbox |
| V6-H3 | Helmet | Tri-feature: dark_ratio + edge_density + saturation |
| V6-H4 | Helmet | Shadow-aware: LAB L-channel normalise before dark_ratio |
| V6-H5 | Helmet | No-person fallback: revert to vehicle-top heuristic |
| V6-H6 | Helmet | Confidence calibration table replacing linear formula |
| V6-V1 | Engine | PARKING_SECS 5->12 (balanced: not too short, not 30s) |
| V6-V2 | Engine | Parking exempt when signal_state==red (at red light) |
| V6-V3 | Engine | Per-violation confirm window avoids cross-type bleed |
| V6-V4 | Engine | Simultaneous violation: each check() fully independent |
| V6-V5 | Engine | Track-age gate raised MIN_TRACK_AGE_FRAMES 5->8 |
| V6-V6 | Engine | Ghost frame budget raised 60->90 for longer occlusions |
| **V7-V1** | **Engine** | **`ev.plate` now refreshes on every update if a better OCR read lands after activation, instead of being frozen at activation time** |

> **Runtime -> T4 GPU** before running.

> **v7 note:** A review claimed the engine "finds the first violation and
> stops evaluating." That was checked directly against `process_frame()`
> and is false — all six violation checks run independently for every
> vehicle every frame; see the v7 section in the final markdown cell for
> the actual root cause of the symptom that prompted the claim.


In [ ]:
import subprocess, sys
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-600:])
    return r.stdout.strip()
print(run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
pkgs = ['ultralytics','easyocr','supervision','opencv-python-headless',
        'Pillow','pandas','matplotlib','seaborn','tqdm','scipy','scikit-learn','lapx']
run(f'{sys.executable} -m pip install -q ' + ' '.join(pkgs))
print('Packages ready')


In [ ]:
import os, cv2, json, shutil, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque, Counter
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Set
from enum import Enum, auto
from tqdm.notebook import tqdm
from PIL import Image
from ultralytics import YOLO
import easyocr
import supervision as sv
warnings.filterwarnings('ignore')

BASE       = Path('/content/traffic_project')
MODELS_DIR = BASE / 'models'
EVIDENCE   = BASE / 'evidence'
CLIPS_DIR  = EVIDENCE / 'clips'
FRAMES_DIR = EVIDENCE / 'frames'
REPORTS    = BASE / 'reports'
for d in [MODELS_DIR, CLIPS_DIR, FRAMES_DIR, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

print('Loading YOLOv8s model...')
master_model = YOLO('yolov8s.pt')
print('Loading EasyOCR...')
ocr_reader   = easyocr.Reader(['en'], gpu=True, verbose=False)
print('Models + OCR ready')

_tracker_yaml_path = MODELS_DIR / 'bytetrack_traffic.yaml'
_tracker_yaml_path.write_text(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.4\n"
    "track_low_thresh: 0.1\n"
    "new_track_thresh: 0.5\n"
    "track_buffer: 60\n"
    "match_thresh: 0.85\n"
    "fuse_score: True\n"
)
TRACKER_NAME = str(_tracker_yaml_path)

COCO = {
    'person': 0, 'bicycle': 1, 'car': 2,
    'motorcycle': 3, 'bus': 5, 'truck': 7,
    'traffic light': 9,
}
VEHICLE_IDS    = {COCO['car'], COCO['motorcycle'], COCO['bus'], COCO['truck'], COCO['bicycle']}
PERSON_ID      = COCO['person']
TLIGHT_ID      = COCO['traffic light']
DETECT_CLASSES = sorted(COCO.values())
EXEMPT_KW      = {'ambulance', 'fire truck', 'police'}

CONF = {
    'detection':     0.45,
    'violation_min': 0.55,
}
NMS_IOU = 0.45
PROCESS_EVERY_N = 2

CLIP_SECS  = 5
CAMERA_ID  = 'CAM_01'

LABEL_COLORS = {
    'Helmet non-compliance':   (0,   0, 255),
    'Triple riding':           (255, 0, 255),
    'Wrong-side driving':      (255, 0,   0),
    'Stop-line violation':     (0, 200, 255),
    'Red-light violation':     (0,   0, 200),
    'Illegal parking':         (128, 0, 255),
}

# Dedup (unchanged from v4)
PLATE_DEDUP_WINDOW_SEC = 120
BBOX_COOLDOWN_SEC      = 3.0
BBOX_COOLDOWN_IOU      = 0.5

# V6-V6: ghost window raised 60->90 for longer occlusions at junctions
GHOST_FRAMES_V6 = 90

# V6-V5: track-age gate raised 5->8 frames (filters more single-frame blips)
MIN_TRACK_AGE_FRAMES  = 8
RED_LIGHT_MIN_DISP_PX = 15
HELMET_MIN_AREA_RATIO = 0.02

# V6-V1: parking dwell raised 5->12s (real parking, not momentary stop)
# V6-V2: parking is also gated on signal_state in engine (not a constant)
PARKING_SECS = 12

print('Setup complete (v6).')


In [ ]:
from google.colab import files
print('Upload your traffic video (.mp4 / .avi / .mov):')
uploaded   = files.upload()
VIDEO_PATH = Path(list(uploaded.keys())[0])

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {VIDEO_PATH}')

TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ret, SAMPLE_FRAME = cap.read()
cap.release()

if not ret or SAMPLE_FRAME is None:
    raise RuntimeError('Could not read first frame.')

print(f'  {VIDEO_PATH.name}  |  {W}x{H}  |  {FPS:.0f} fps  |  {TOTAL_FRAMES} frames  |  {TOTAL_FRAMES/FPS:.0f}s')
FRAME_AREA = W * H


In [ ]:
USE_FULL_FRAME  = True
USE_PERSPECTIVE = True
PERSPECTIVE_MIN_AREA_RATIO = 0.012

if USE_FULL_FRAME:
    horizon_y  = int(H * 0.45)
    horizon_x1 = int(W * 0.20)
    horizon_x2 = int(W * 0.80)
    ROI_POLYGON = np.array([
        [0, H], [W, H], [horizon_x2, horizon_y], [horizon_x1, horizon_y],
    ], dtype=np.int32)
    print('Using road-trapezoid ROI.')
else:
    roi_points = []; roi_img = SAMPLE_FRAME.copy()
    def _mouse_cb(event, x, y, flags, param):
        global roi_img
        if event == cv2.EVENT_LBUTTONDOWN:
            roi_points.append((x, y))
            cv2.circle(roi_img, (x, y), 5, (0,255,0), -1)
            if len(roi_points) > 1:
                cv2.line(roi_img, roi_points[-2], roi_points[-1], (0,255,0), 2)
            cv2.imshow('Draw ROI', roi_img)
    cv2.imshow('Draw ROI', roi_img)
    cv2.setMouseCallback('Draw ROI', _mouse_cb)
    cv2.waitKey(0); cv2.destroyAllWindows()
    ROI_POLYGON = np.array(roi_points, dtype=np.int32)

def in_roi(cx, cy):
    if ROI_POLYGON.shape[0] < 3: return True
    poly = ROI_POLYGON.reshape(-1,1,2).astype(np.int32)
    return cv2.pointPolygonTest(poly, (float(cx), float(cy)), False) >= 0

def clamp_bbox(bbox, w, h):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(int(x1), w-1)); y1 = max(0, min(int(y1), h-1))
    x2 = max(x1+1, min(int(x2), w)); y2 = max(y1+1, min(int(y2), h))
    return x1, y1, x2, y2

def perspective_ok(bbox):
    if not USE_PERSPECTIVE or FRAME_AREA == 0: return True
    x1, y1, x2, y2 = bbox
    return ((float(x2)-float(x1))*(float(y2)-float(y1))/FRAME_AREA) >= PERSPECTIVE_MIN_AREA_RATIO

def detect_stop_line(frame):
    h, w = frame.shape[:2]
    search_top = int(h * 0.65)
    roi   = frame[search_top:, :]
    gray  = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 30, 100)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=60,
                             minLineLength=int(w*0.25), maxLineGap=40)
    if lines is None: return int(h*0.80)
    h_lines = []
    for line in lines:
        x1,y1_l,x2,y2_l = line[0]
        angle = abs(np.degrees(np.arctan2(y2_l-y1_l, x2-x1)))
        if angle < 8 or angle > 172: h_lines.append(y1_l)
    if not h_lines: return int(h*0.80)
    stop_y = search_top + int(np.max(h_lines))
    return max(int(h*0.65), min(int(h*0.92), stop_y))

STOP_LINE_Y = detect_stop_line(SAMPLE_FRAME)
PARKING_EXCLUSION_BAND = 80
FORBIDDEN_PARKING_ZONES = [(0, 0, W, STOP_LINE_Y - PARKING_EXCLUSION_BAND)]

def in_forbidden_zone(cx, cy):
    for (x1,y1,x2,y2) in FORBIDDEN_PARKING_ZONES:
        if x1 <= cx <= x2 and y1 <= cy <= y2: return True
    return False

vis = SAMPLE_FRAME.copy()
cv2.line(vis, (0, STOP_LINE_Y), (W, STOP_LINE_Y), (0,255,255), 2)
cv2.polylines(vis, [ROI_POLYGON], True, (0,255,0), 2)
plt.figure(figsize=(13,6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Green=ROI | Cyan=stop line y={STOP_LINE_Y}')
plt.axis('off'); plt.tight_layout(); plt.show()
print(f'Stop line y={STOP_LINE_Y}')


In [ ]:
class ViolationState(Enum):
    NEW      = auto()
    ACTIVE   = auto()
    RESOLVED = auto()

@dataclass
class ViolationEvent:
    event_id:     str
    track_id:     int
    violation:    str
    state:        ViolationState = ViolationState.NEW
    plate:        str = 'UNKNOWN'
    confidence:   float = 0.0
    camera:       str = 'CAM_01'
    first_frame:  int = 0
    last_frame:   int = 0
    first_ts:     str = ''
    last_ts:      str = ''
    bbox:         Tuple = (0, 0, 0, 0)
    vehicle_type: str = ''
    rider_count:  int = 0
    exempt:       bool = False
    evidence_img: str = ''
    evidence_clip: str = ''

    def activate(self, frame_idx, ts, conf, bbox):
        self.state = ViolationState.ACTIVE
        self.first_frame = frame_idx; self.last_frame = frame_idx
        self.first_ts = ts; self.last_ts = ts
        self.confidence = conf; self.bbox = bbox

    def update(self, frame_idx, ts, conf):
        self.last_frame = frame_idx; self.last_ts = ts
        self.confidence = max(self.confidence, conf)

    def resolve(self):
        self.state = ViolationState.RESOLVED

    def to_dict(self):
        d = asdict(self); d['state'] = self.state.name; return d


class ANPRRegistry:
    """Temporal voting: mode of valid reads per track."""
    def __init__(self):
        self.records: List[Dict] = []
        self._reads: Dict[int, List[Tuple[str, float]]] = defaultdict(list)

    def log(self, track_id, plate, ocr_conf, frame_idx, ts, camera):
        self._reads[track_id].append((plate, ocr_conf))
        self.records.append({'track_id': track_id, 'plate': plate,
                             'ocr_confidence': ocr_conf, 'frame': frame_idx,
                             'timestamp': ts, 'camera': camera})

    def best_plate(self, track_id):
        """
        V7-O4: confidence-weighted voting instead of pure frequency count.
        Pure Counter.most_common() treats every read as equally trustworthy,
        so three low-confidence reads of one (possibly OCR-confused) string
        could outvote a single high-confidence correct read. Each candidate
        plate's reads are summed by confidence instead of counted by
        occurrence; the plate with the highest total confidence wins.
        """
        reads = self._reads.get(track_id, [])
        valid = [(p, c) for p, c in reads if p != 'UNKNOWN']
        if len(valid) < 2:
            return 'UNKNOWN'
        conf_totals: Dict[str, float] = defaultdict(float)
        for p, c in valid:
            conf_totals[p] += c
        return max(conf_totals, key=conf_totals.get)

    def to_df(self):
        return pd.DataFrame(self.records) if self.records else pd.DataFrame()

print('Data classes defined (v6: ANPRRegistry requires 2 valid reads).')


In [ ]:
import re
import math

# =====================================================================
# SIGNAL TRACKER  (unchanged from v4)
# =====================================================================
class SignalTracker:
    REDETECT_INTERVAL = 120
    CYCLE = {'red': 60, 'green': 45, 'yellow': 5}

    def __init__(self, fps):
        self.fps = fps
        self._tracker = None; self._tbox_xywh = None
        self._last_detect = -(self.REDETECT_INTERVAL + 1)
        self.current = 'unknown'
        self.history: deque = deque(maxlen=1000)
        self._total_cycle = sum(self.CYCLE.values())
        self._cycle_start_frame = 0

    def _area_threshold(self, crop):
        return max(20, int(crop.shape[0] * crop.shape[1] * 0.05))

    def _classify_crop(self, crop):
        if crop is None or crop.size == 0: return 'unknown'
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        r1 = cv2.inRange(hsv, np.array([0,   100, 80]), np.array([10,  255, 255]))
        r2 = cv2.inRange(hsv, np.array([160, 100, 80]), np.array([180, 255, 255]))
        g  = cv2.inRange(hsv, np.array([40,  100, 80]), np.array([90,  255, 255]))
        y  = cv2.inRange(hsv, np.array([15,  100, 80]), np.array([40,  255, 255]))
        thr = self._area_threshold(crop)
        px = {'red': cv2.countNonZero(r1|r2), 'green': cv2.countNonZero(g), 'yellow': cv2.countNonZero(y)}
        best = max(px, key=px.get)
        return best if px[best] >= thr else 'unknown'

    def _try_init_tracker(self, frame, bbox_xyxy):
        x1,y1,x2,y2 = [int(v) for v in bbox_xyxy]
        w,h = x2-x1, y2-y1
        if w<=0 or h<=0: return False
        try:
            self._tracker = cv2.TrackerCSRT_create()
            self._tbox_xywh = (x1,y1,w,h)
            self._tracker.init(frame, self._tbox_xywh); return True
        except AttributeError:
            self._tracker = None; self._tbox_xywh = (x1,y1,w,h); return False

    def update(self, frame, frame_idx, detected_tlight_boxes):
        need_redetect = (self._tracker is None or
                         (frame_idx - self._last_detect) >= self.REDETECT_INTERVAL)
        if need_redetect:
            self._last_detect = frame_idx
            if detected_tlight_boxes:
                best_box = max(detected_tlight_boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]))
                self._try_init_tracker(frame, best_box)
        if self._tracker is not None:
            ok, bbox_xywh = self._tracker.update(frame)
            if ok:
                x,y,w,h = [int(v) for v in bbox_xywh]
                crop = frame[max(0,y):y+h, max(0,x):x+w]
                state = self._classify_crop(crop)
                self.history.append((frame_idx, state))
                self.current = state; return self.current
            else:
                self._tracker = None
        elif self._tbox_xywh is not None:
            x,y,w,h = self._tbox_xywh
            crop = frame[max(0,y):y+h, max(0,x):x+w]
            state = self._classify_crop(crop)
            if state != 'unknown':
                self.history.append((frame_idx, state)); self.current = state; return self.current
        if len(self.history) > 20:
            if self._cycle_start_frame == 0:
                self._cycle_start_frame = self.history[0][0]
            elapsed_s = (frame_idx - self._cycle_start_frame) / self.fps
            t = elapsed_s % self._total_cycle; acc = 0
            for phase, dur in self.CYCLE.items():
                acc += dur
                if t < acc: self.current = phase; return self.current
        return 'unknown'

    @property
    def is_red(self): return self.current == 'red'


# =====================================================================
# LANE DIRECTION TRACKER  (unchanged from v4)
# =====================================================================
class LaneDirectionTracker:
    LEARN_FRAMES = 300; MAX_CLUSTERS = 4
    ANGLE_TOL = 50.0; DIR_WINDOW = 20; MIN_DISP = 10

    def __init__(self):
        self.tracks: Dict[int, List] = defaultdict(list)
        self.learned = False; self._centers = None
        self._frame_n = 0; self._last_fi = -1; self._n_clusters = 0

    def update(self, track_id, cx, cy, frame_idx=-1):
        self.tracks[track_id].append((cx, cy))
        if frame_idx != self._last_fi:
            self._frame_n += 1; self._last_fi = frame_idx
        if not self.learned and self._frame_n >= self.LEARN_FRAMES:
            self._learn()

    def _learn(self):
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score
        dirs = []
        for pts in self.tracks.values():
            if len(pts) < 6: continue
            arr = np.array(pts); dx = arr[-1,0]-arr[0,0]; dy = arr[-1,1]-arr[0,1]
            n = np.hypot(dx,dy)
            if n > self.MIN_DISP: dirs.append([dx/n, dy/n])
        if len(dirs) < 2: return
        X = np.array(dirs); best_k = 1; best_score = -1.0
        for k in range(2, min(self.MAX_CLUSTERS+1, len(dirs))):
            km = KMeans(n_clusters=k, n_init=10, random_state=42)
            lbl = km.fit_predict(X)
            if len(set(lbl)) < 2: continue
            try: s = silhouette_score(X, lbl)
            except Exception: continue
            if s > best_score: best_score = s; best_k = k
        km = KMeans(n_clusters=best_k, n_init=10, random_state=42); km.fit(X)
        self._centers = km.cluster_centers_; self._n_clusters = best_k
        self.learned = True
        print(f'  [Lane] k={best_k} clusters (silhouette={best_score:.2f})')

    def _direction(self, track_id):
        pts = self.tracks.get(track_id, [])
        if len(pts) < self.DIR_WINDOW: return None
        arr = np.array(pts[-self.DIR_WINDOW:])
        dx = arr[-1,0]-arr[0,0]; dy = arr[-1,1]-arr[0,1]
        n = np.hypot(dx,dy)
        if n < self.MIN_DISP: return None
        return np.array([dx/n, dy/n])

    def is_wrong_side(self, track_id):
        if not self.learned: return False
        d = self._direction(track_id)
        if d is None: return False
        sims = self._centers @ d; best_sim = float(np.max(sims))
        return best_sim < np.cos(np.radians(self.ANGLE_TOL))


# =====================================================================
# V6-O: OCR -- detect_and_ocr_plate  (complete rewrite of helper)
# Changes:
#   V6-O1: OCR_INTERVAL 50->25
#   V6-O2: dual-crop: primary bottom-25% + fallback bottom-40%
#   V6-O3: target 64px crop height
#   V6-O4: CLAHE on L-channel before thresholding
#   V6-O5: paragraph=False, min_size=8
#   V6-O6: length-weighted confidence
#   V6-O7: dual-polarity (normal + inverted), pick higher-confidence result
#   V6-O8: min-reads gate moved into ANPRRegistry.best_plate()
#   V6-O9: removed redundant avg_conf double-gate; kept format gate
# =====================================================================
OCR_INTERVAL   = 25      # V6-O1: was 50; every ~1s at 25fps with PROCESS_EVERY_N=2
OCR_CONF_FLOOR = 0.40
PLATE_REGEX    = re.compile(r'^[A-HJ-NP-RT-Z]{2}\d{1,2}[A-HJ-NP-RT-Z]{0,3}\d{4}$')
# V7-O5: Indian plates never use I, O, or Q in the letter positions
# specifically because the issuing authority avoids exactly the 1/0
# confusion this system has to deal with. The old regex used a plain
# [A-Z] letter class, which meant a misread "1" that OCR rendered as
# "I" (a structurally valid letter) PASSED format validation unchanged
# -- silently keeping a wrong plate instead of triggering the
# confusion-correction step below. Excluding I/O/Q from the letter
# classes makes that misread correctly FAIL validation, so
# _try_correct_plate_format() actually gets a chance to fix it.

_ocr_last_frame: Dict[int, int] = {}
_ocr_cache:      Dict[int, str] = {}
_ocr_clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(4, 4))   # V6-O4


def _validate_plate_format(plate: str) -> bool:
    return bool(PLATE_REGEX.match(plate))


# V7-O2: common OCR character confusions, used to attempt a correction
# when a read is ONE character away from matching PLATE_REGEX. This
# targets a real, common OCR failure mode (visually similar characters),
# not arbitrary fuzzy matching -- only single-character substitutions
# from this fixed confusion table are tried, and only when they turn an
# invalid-format read into a valid-format one.
_OCR_CONFUSION_PAIRS = {
    'O': '0', '0': 'O',
    'I': '1', '1': 'I',
    'S': '5', '5': 'S',
    'B': '8', '8': 'B',
    'Z': '2', '2': 'Z',
    'G': '6', '6': 'G',
    'D': '0', 'Q': '0',
}


def _try_correct_plate_format(plate: str) -> str:
    """
    V7-O2: If `plate` does not match PLATE_REGEX, try every single-position
    substitution from _OCR_CONFUSION_PAIRS and return the first result that
    DOES match. This recovers reads that are correct except for one
    visually-confusable character, instead of discarding them outright.
    Returns the original string unchanged if no single substitution fixes it
    (does not attempt multi-character correction -- that would risk
    inventing a plausible-looking but wrong plate).
    """
    if _validate_plate_format(plate):
        return plate
    for i, ch in enumerate(plate):
        sub = _OCR_CONFUSION_PAIRS.get(ch)
        if sub is None:
            continue
        candidate = plate[:i] + sub + plate[i+1:]
        if _validate_plate_format(candidate):
            return candidate
    return plate


def _preprocess_plate_crop(crop_bgr: np.ndarray) -> np.ndarray:
    """
    V6-O3+O4: Resize to ~64px height, then CLAHE on L-channel,
    then bilateral denoise, then dual-threshold.
    Returns thresholded grayscale ready for EasyOCR.
    """
    h, w = crop_bgr.shape[:2]
    if h == 0 or w == 0:
        return None
    # Target 64px height (was 120//h which is uncapped and inconsistent)
    scale  = max(1.0, 64.0 / h)
    new_w  = max(1, int(w * scale))
    new_h  = max(1, int(h * scale))
    resized = cv2.resize(crop_bgr, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    # V6-O4: CLAHE on LAB L-channel to lift low-contrast / shadowed plates
    lab = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = _ocr_clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

    gray     = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    denoised = cv2.bilateralFilter(gray, 9, 60, 60)
    _, thr   = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    thr      = cv2.morphologyEx(thr, cv2.MORPH_CLOSE, kernel)
    return thr


def _run_ocr(img: np.ndarray) -> Tuple[str, float]:
    """
    V6-O5: paragraph=False, min_size=8 for character-level results.
    V6-O6: length-weighted confidence (longer fragments count more).
    V6-O7: run on normal + inverted image; return whichever gives higher conf.
    Returns (plate_text, weighted_conf).
    """
    best_plate = 'UNKNOWN'
    best_conf  = 0.0

    for img_variant in [img, cv2.bitwise_not(img)]:   # V6-O7: dual polarity
        try:
            results = ocr_reader.readtext(
                img_variant, detail=1,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                paragraph=False,      # V6-O5
                min_size=8,           # V6-O5
            )
        except Exception:
            continue

        if not results:
            continue

        # V6-O6: length-weighted confidence
        total_len  = sum(max(1, len(t)) for (_, t, _) in results)
        wconf      = sum((len(t) / total_len) * c for (_, t, c) in results) if total_len else 0.0
        parts      = [t for (_, t, c) in results if c >= OCR_CONF_FLOOR]
        plate_cand = ''.join(parts).upper().replace(' ', '')
        plate_cand = plate_cand[:10] if plate_cand else 'UNKNOWN'

        # V7-O2: try single-character confusion correction before
        # rejecting an otherwise-plausible read outright.
        if plate_cand != 'UNKNOWN':
            plate_cand = _try_correct_plate_format(plate_cand)
        if plate_cand != 'UNKNOWN' and not _validate_plate_format(plate_cand):
            plate_cand = 'UNKNOWN'

        # Keep the higher-confidence polarity result
        if wconf > best_conf:
            best_conf  = wconf
            best_plate = plate_cand

    return best_plate, best_conf


def detect_and_ocr_plate(frame: np.ndarray,
                          vehicle_bbox: Tuple,
                          track_id: int,
                          frame_idx: int,
                          anpr: 'ANPRRegistry' = None,
                          ts: str = '',
                          vehicle_class: str = 'car') -> str:
    """
    V6-O1-O9 + V7-O1: Improved OCR pipeline.
    Throttled to OCR_INTERVAL. Returns per-call plate for display;
    authoritative plate is anpr.best_plate(track_id).

    V7-O1: crop band is now selected by vehicle_class instead of a single
    fixed bottom-% for every vehicle type. Root cause this fixes: a bus/
    truck bounding box is much taller than a motorcycle's, so a fixed
    "bottom 25%" band that works for a motorcycle plate lands on the
    wheel well or road surface on a bus/truck, well below where the
    plate actually is mounted. Motorcycle plates also commonly sit
    higher relative to the bbox bottom than car plates do.
    """
    if track_id in _ocr_last_frame:
        if frame_idx - _ocr_last_frame[track_id] < OCR_INTERVAL:
            return _ocr_cache.get(track_id, 'UNKNOWN')

    x1, y1, x2, y2 = [int(v) for v in vehicle_bbox]
    h = y2 - y1
    w_v = x2 - x1

    best_plate = 'UNKNOWN'
    best_conf  = 0.0

    # V7-O1: per-vehicle-type crop bands.
    # Each tuple is (top_fraction_of_h, bottom_fraction_of_h) measured
    # from the TOP of the bbox (y1), in the order to try them.
    # - motorcycle: plate sits higher up (under the seat/tail), and the
    #   bbox bottom often includes the rear wheel/road shadow.
    # - bus/truck: plate is mounted on a cab/bumper that occupies a much
    #   smaller fraction of the overall (tall) bbox height; searching the
    #   bottom 25% on a truck usually misses it entirely.
    # - car/default: bottom 25%/40% bands from v6 remain a reasonable fit.
    crop_bands_by_class = {
        'motorcycle': [(0.45, 0.75), (0.30, 0.85)],
        'bus':        [(0.55, 0.85), (0.40, 0.95)],
        'truck':      [(0.55, 0.85), (0.40, 0.95)],
        'car':        [(0.75, 1.00), (0.60, 1.00)],
        'bicycle':    [(0.45, 0.75), (0.30, 0.85)],
    }
    bands = crop_bands_by_class.get(vehicle_class, crop_bands_by_class['car'])
    crop_configs = [
        (int(h * top_frac), y1 + int(h * bot_frac))
        for (top_frac, bot_frac) in bands
    ]

    for (cy1_offset, cy2) in crop_configs:
        plate_y1   = y1 + cy1_offset
        plate_crop = frame[max(0, plate_y1):cy2, max(0, x1):x2]

        if plate_crop.ndim < 2 or plate_crop.shape[0] < 4 or plate_crop.shape[1] < 8:
            continue

        proc = _preprocess_plate_crop(plate_crop)
        if proc is None:
            continue

        plate, conf = _run_ocr(proc)

        # V7-O3: previously broke out of the loop on the FIRST crop band
        # that returned any valid-format plate, even at low confidence,
        # so a clearer read from the fallback band never got a chance to
        # win. Now both bands are tried and compared by confidence; we
        # only skip the fallback early if the primary read was already
        # high-confidence (>=0.65), since in that case the fallback band
        # is unlikely to do better and running it just costs GPU time.
        if conf > best_conf:
            best_conf  = conf
            best_plate = plate

        if best_plate != 'UNKNOWN' and best_conf >= 0.65:
            break   # primary crop succeeded with high confidence; skip fallback

    # V6-O9 + V7-O2: format gate with confusion correction.
    # (conf is already embedded in the format-valid result above;
    #  double-gating on avg_conf was nullifying valid reads whose
    #  average was dragged down by a short low-conf fragment)
    if best_plate != 'UNKNOWN':
        best_plate = _try_correct_plate_format(best_plate)
    if best_plate != 'UNKNOWN' and not _validate_plate_format(best_plate):
        best_plate = 'UNKNOWN'

    _ocr_last_frame[track_id] = frame_idx
    _ocr_cache[track_id]      = best_plate

    if anpr is not None:
        anpr.log(track_id, best_plate, float(best_conf), frame_idx, ts, CAMERA_ID)

    return best_plate


# =====================================================================
# V6-H: HELMET DETECTION  (complete rewrite of _has_helmet)
# Changes:
#   V6-H1: person-bbox IoU matching for head extraction
#   V6-H2: top 30% of matched person bbox as head crop
#   V6-H3: tri-feature: dark_ratio + edge_density + saturation
#   V6-H4: shadow-aware LAB L-channel normalise before dark_ratio
#   V6-H5: no-person fallback to vehicle-top heuristic
#   V6-H6: confidence calibration table
# =====================================================================

def _iou_scalar(a, b) -> float:
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1=max(ax1,bx1); iy1=max(ay1,by1); ix2=min(ax2,bx2); iy2=min(ay2,by2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    if inter == 0: return 0.0
    ua = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/ua if ua > 0 else 0.0


def _helmet_features(head_region: np.ndarray) -> Tuple[float, float, float, bool]:
    """
    Returns (dark_ratio, edge_density, saturation_mean, edge_density_reliable).

    dark_ratio:      fraction of pixels below L=80 after LAB normalise (helmet = dark blob)
    edge_density:    Canny edge pixel fraction in head region (helmet = smooth, not spiky hair)
    saturation_mean: mean HSV S-channel (helmet = low sat for black helmets; hair = variable)
    edge_density_reliable: V7-H1 -- False when the crop is too small in pixels
        for Canny to resolve hair-strand texture at all. Below ~25x25px,
        dark hair routinely measures as LOW edge_density purely from
        insufficient resolution, not because it is actually smooth like a
        helmet shell. Treating that as "smooth = helmet" was the root
        cause of bare heads being classified as compliant.
    """
    if head_region.size == 0:
        return 0.0, 0.0, 0.0, False

    hr_h, hr_w = head_region.shape[:2]
    # V7-H1: resolution floor for edge_density to be meaningful.
    edge_density_reliable = (hr_h >= 25 and hr_w >= 25)

    lab = cv2.cvtColor(head_region, cv2.COLOR_BGR2LAB)
    l_ch = lab[:,:,0].astype(np.float32)
    l_min, l_max = l_ch.min(), l_ch.max()
    if l_max > l_min:
        l_norm = ((l_ch - l_min) / (l_max - l_min) * 255).astype(np.uint8)
    else:
        l_norm = l_ch.astype(np.uint8)

    dark_ratio = float(np.sum(l_norm < 80) / max(1, l_norm.size))

    gray         = cv2.cvtColor(head_region, cv2.COLOR_BGR2GRAY)
    blurred_g    = cv2.GaussianBlur(gray, (3,3), 0)
    edges        = cv2.Canny(blurred_g, 40, 120)
    edge_density = float(np.sum(edges > 0) / max(1, edges.size))

    hsv  = cv2.cvtColor(head_region, cv2.COLOR_BGR2HSV)
    sat_mean = float(np.mean(hsv[:,:,1])) / 255.0

    return dark_ratio, edge_density, sat_mean, edge_density_reliable


def _helmet_confidence(dark_ratio: float, edge_density: float,
                        sat_mean: float, edge_density_reliable: bool) -> Tuple[bool, float]:
    """
    V7-H1: rebuilt decision boundary. The v6 version required dark_ratio
    < 0.20 to flag "no helmet" and otherwise defaulted toward "helmet" for
    any dark, low-edge-density blob. At typical CCTV head-crop resolution,
    dark hair routinely produces dark_ratio 0.25-0.45 AND low edge_density
    (because Canny can't resolve hair strands at that size, not because
    the surface is actually smooth) -- so real bare heads were being
    classified as wearing a helmet.

    New approach:
      - edge_density is ONLY used as helmet evidence when the crop is
        large enough for it to be meaningful (edge_density_reliable).
        When unreliable, the decision rests on dark_ratio + saturation
        alone, with a HIGHER bar for "helmet" since we have less evidence.
      - The "no helmet" bar is raised from <0.20 to <0.35 dark_ratio --
        most bare-head crops (skin + lit hair) fall under this after LAB
        stretch; true helmet shells (especially black, full-face) are
        consistently >0.45 dark_ratio with low saturation.
      - Default for genuinely ambiguous cases is now "no helmet" (flag
        for review) rather than "has helmet" (silently compliant) --
        a missed violation is worse than a flagged false positive that
        a human reviewer can dismiss from the evidence image.
    """
    # Strong helmet signature: very dark, low saturation, AND (if we can
    # trust it) smooth surface. Full-face / half helmets, most colours.
    if dark_ratio > 0.55 and sat_mean < 0.25:
        if not edge_density_reliable or edge_density < 0.12:
            conf = 0.72 + 0.15 * min(1.0, (dark_ratio - 0.55) / 0.25)
            return True, min(0.90, conf)

    # Moderate helmet signature: needs the smoothness check AND enough
    # resolution to trust it, since this band overlaps with dark hair.
    if 0.40 < dark_ratio <= 0.55 and sat_mean < 0.30:
        if edge_density_reliable and edge_density < 0.10:
            return True, 0.65
        # Unreliable edge signal in this overlap band -> don't trust it,
        # fall through to the no-helmet checks below.

    # No-helmet: bright/skin-toned region clearly visible (raised from 0.20)
    if dark_ratio < 0.35:
        conf = 0.62 + 0.18 * min(1.0, (0.35 - dark_ratio) / 0.35)
        return False, min(0.90, conf)

    # High saturation in the dark band = patterned cloth/turban/coloured
    # hair accessory, not a helmet shell (helmets are low-sat almost always)
    if dark_ratio >= 0.35 and sat_mean >= 0.30:
        return False, 0.58

    # Genuinely ambiguous (dark_ratio 0.35-0.55, low sat, unreliable edges):
    # default to flagging rather than silently passing as compliant.
    return False, 0.50


def has_helmet_v6(frame: np.ndarray,
                  vehicle_bbox: Tuple,
                  person_boxes: List[np.ndarray]) -> Tuple[bool, float]:
    """
    V6-H1-H5 + V7-H1: improved helmet detector.
    1. Try to find a person box that significantly overlaps the top half
       of the motorcycle bbox (that is the rider on the seat).
    2. Extract the top 30% of that person bbox as the head region.
    3. Run tri-feature analysis (edge_density gated by resolution).
    4. If no overlapping person found, fall back to vehicle-top heuristic.
    """
    x1, y1, x2, y2 = [int(v) for v in vehicle_bbox]
    vh = y2 - y1
    vw = x2 - x1
    if vh <= 0 or vw <= 0:
        return False, 0.5   # V7-H1: can't classify -> flag for review, don't silently pass

    rider_zone = (x1, y1, x2, y1 + int(vh * 0.60))
    best_person = None
    best_iou    = 0.15

    for pb in person_boxes:
        px1,py1,px2,py2 = [int(v) for v in pb]
        iou = _iou_scalar(rider_zone, (px1,py1,px2,py2))
        if iou > best_iou:
            best_iou    = iou
            best_person = (px1, py1, px2, py2)

    if best_person is not None:
        px1, py1, px2, py2 = best_person
        ph = py2 - py1
        head_y2     = py1 + max(8, int(ph * 0.30))
        head_region = frame[max(0,py1):min(frame.shape[0],head_y2),
                            max(0,px1):min(frame.shape[1],px2)]
    else:
        head_y2     = y1 + max(8, int(vh * 0.28))
        head_region = frame[max(0,y1):max(0,head_y2), max(0,x1):max(0,x2)]

    if head_region.size == 0:
        return False, 0.5   # V7-H1: was True (silently compliant); now flags for review

    # V7-H1: minimum size guard raised. Below this, even dark_ratio is
    # too noisy to trust (a handful of pixels), so flag for review
    # instead of asserting compliance.
    if head_region.shape[0] < 10 or head_region.shape[1] < 10:
        return False, 0.5

    dark_r, edge_d, sat_m, edge_reliable = _helmet_features(head_region)
    return _helmet_confidence(dark_r, edge_d, sat_m, edge_reliable)


# =====================================================================
# BBOX SMOOTHER  (unchanged from v4)
# =====================================================================
class BBoxSmoother:
    ALPHA = 0.6
    def __init__(self): self._state: Dict[int, np.ndarray] = {}
    def smooth(self, track_id, bbox):
        new = np.array(bbox, dtype=np.float32)
        if track_id not in self._state: self._state[track_id] = new
        else: self._state[track_id] = self.ALPHA*new + (1-self.ALPHA)*self._state[track_id]
        x1,y1,x2,y2 = self._state[track_id]
        return int(x1), int(y1), int(x2), int(y2)
    def drop(self, track_id): self._state.pop(track_id, None)


# =====================================================================
# ADAPTIVE ENHANCER  (unchanged from v4)
# =====================================================================
class AdaptiveEnhancer:
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        self.stats = defaultdict(int)
    def enhance(self, frame):
        applied = []; out = frame.copy()
        gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
        if np.mean(gray) < 60:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB); l,a,b = cv2.split(lab)
            out = cv2.cvtColor(cv2.merge([self.clahe.apply(l),a,b]), cv2.COLOR_LAB2BGR)
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
            applied.append('low_light'); self.stats['low_light'] += 1
        if cv2.Laplacian(gray, cv2.CV_64F).var() < 80:
            blurred = cv2.GaussianBlur(out,(0,0),3)
            out = cv2.addWeighted(out,1.5,blurred,-0.5,0)
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
            applied.append('deblur'); self.stats['deblur'] += 1
        if np.mean(gray) > 160 and np.std(gray) < 30:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB); l,a,b = cv2.split(lab)
            l = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)
            out = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)
            applied.append('dehaze'); self.stats['dehaze'] += 1
        self.stats['total'] += 1
        return out, applied

print('All helpers defined (v6: improved OCR + helmet detector).')


In [ ]:
class ViolationEngineV6:
    """
    V6 changes from v4:
      V6-V1: PARKING_SECS 5->12
      V6-V2: parking exempt during red signal (at red light = waiting, not illegally parked)
      V6-V3: per-violation-type confirm counters are already the case in v4 via key=(tid,vtype)
             -- V6-V3 adds explicit reset on CONFIRM_RESET_GAP per key correctly
      V6-V4: each check() is independent (already was in v4; V6 makes _flag non-caching)
      V6-V5: MIN_TRACK_AGE_FRAMES 5->8 (set in Cell 2)
      V6-V6: GHOST_FRAMES 60->90 (set in Cell 2)
      V6-H*: has_helmet_v6() replaces _has_helmet()
      V6-O*: detect_and_ocr_plate() upgraded in Cell 6
    """

    CONFIRM_FRAMES    = 4
    CONFIRM_RESET_GAP = 20
    GHOST_FRAMES      = GHOST_FRAMES_V6   # V6-V6: 90

    def __init__(self, fps, frame_w, frame_h):
        self.fps = fps; self.W = frame_w; self.H = frame_h
        self.signal   = SignalTracker(fps)
        self.lane     = LaneDirectionTracker()
        self.anpr     = ANPRRegistry()
        self.enh      = AdaptiveEnhancer()
        self.smoother = BBoxSmoother()

        self._events:       Dict[Tuple, ViolationEvent] = {}
        self._confirm:      Dict[Tuple, int]            = defaultdict(int)
        self._last_flagged: Dict[Tuple, int]            = {}
        self._last_seen:    Dict[int, int]              = {}
        self._first_seen:   Dict[int, int]              = {}

        self._plate_violations: Dict[str, Dict[str, int]] = defaultdict(dict)
        self._bbox_cooldown:    Dict[str, List[Tuple[int,Tuple]]] = defaultdict(list)
        self._plate_to_track:   Dict[str, int] = {}

        self.resolved_events: List[ViolationEvent] = []
        self._ev_counter = 0

    # ── helpers ──────────────────────────────────────────────────────────────
    def _ts(self, fi): return datetime.utcfromtimestamp(fi/self.fps).strftime('%H:%M:%S.%f')[:-3]
    def _new_id(self, fi):
        self._ev_counter += 1; return f'VIO-{CAMERA_ID}-{fi:06d}-{self._ev_counter:04d}'

    def _fuse(self, *scores):
        arr = [s for s in scores if s > 0]
        return float(np.prod(arr)**(1.0/len(arr))) if arr else 0.0

    def _iou(self, a, b):
        ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
        ix1=max(ax1,bx1); iy1=max(ay1,by1); ix2=min(ax2,bx2); iy2=min(ay2,by2)
        inter=max(0,ix2-ix1)*max(0,iy2-iy1)
        if inter==0: return 0.0
        ua=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/ua if ua>0 else 0.0

    def _riders(self, vbbox, person_boxes):
        return sum(1 for pb in person_boxes if self._iou(vbbox, pb) > 0.15)

    def _plate_dedup_blocked(self, plate, vtype, frame_idx):
        if plate == 'UNKNOWN': return False
        last = self._plate_violations[plate].get(vtype)
        if last is None: return False
        return (frame_idx - last) / self.fps < PLATE_DEDUP_WINDOW_SEC

    def _bbox_cooldown_blocked(self, vtype, bbox, frame_idx):
        window_frames = int(BBOX_COOLDOWN_SEC * self.fps / PROCESS_EVERY_N)
        recent = self._bbox_cooldown[vtype]
        recent[:] = [(f,b) for (f,b) in recent if frame_idx-f <= window_frames]
        return any(self._iou(b, bbox) > BBOX_COOLDOWN_IOU for (f,b) in recent)

    def _register_activation(self, plate, vtype, bbox, frame_idx):
        if plate != 'UNKNOWN': self._plate_violations[plate][vtype] = frame_idx
        self._bbox_cooldown[vtype].append((frame_idx, bbox))

    # ── event lifecycle ───────────────────────────────────────────────────────
    def _flag(self, key, frame_idx, ts, conf, bbox, track_id,
              vehicle_type, rider_count, exempt, plate):
        vtype = key[1]

        if key not in self._events:
            if self._plate_dedup_blocked(plate, vtype, frame_idx): return False
            if self._bbox_cooldown_blocked(vtype, bbox, frame_idx): return False

        last_f = self._last_flagged.get(key, -999)
        # V6-V3: reset confirm if gap exceeded (same as v4 but explicit per-key)
        if frame_idx - last_f > self.CONFIRM_RESET_GAP:
            self._confirm[key] = 0

        self._confirm[key]     += 1
        self._last_flagged[key] = frame_idx

        if key not in self._events:
            self._events[key] = ViolationEvent(
                event_id=self._new_id(frame_idx), track_id=track_id,
                violation=vtype, camera=CAMERA_ID,
                vehicle_type=vehicle_type, rider_count=rider_count,
                exempt=exempt, plate=plate,
            )

        ev = self._events[key]
        if ev.state == ViolationState.NEW:
            if self._confirm[key] >= self.CONFIRM_FRAMES:
                ev.activate(frame_idx, ts, conf, bbox)
                self._register_activation(plate, vtype, bbox, frame_idx)
                if plate != 'UNKNOWN': self._plate_to_track[plate] = track_id
                return True
        elif ev.state == ViolationState.ACTIVE:
            ev.update(frame_idx, ts, conf)
            # V7-V1: refresh plate on every update, not just at activation.
            # Root cause fixed: OCR is throttled (OCR_INTERVAL frames per
            # track), so the plate available at the EXACT frame a violation
            # activates is often still 'UNKNOWN' even though a later OCR
            # read on the SAME track succeeds before the event resolves.
            # Previously ev.plate was set once at activate() and never
            # touched again, so a real plate read afterward was discarded.
            # Only overwrite if we have something better than what's stored,
            # so a later transient OCR failure doesn't erase a good read.
            if plate != 'UNKNOWN' and ev.plate == 'UNKNOWN':
                ev.plate = plate
                self._plate_to_track[plate] = track_id

        return False

    def _resolve_stale(self, active_ids, frame_idx):
        for tid in list(self._last_seen):
            if tid not in active_ids:
                if frame_idx - self._last_seen[tid] > self.GHOST_FRAMES:
                    for key in [k for k in list(self._events) if k[0]==tid]:
                        ev = self._events.pop(key, None)
                        if ev and ev.state == ViolationState.ACTIVE:
                            ev.resolve(); self.resolved_events.append(ev)
                    self._last_seen.pop(tid, None)
                    self._first_seen.pop(tid, None)
                    self.smoother.drop(tid)

    # ── main frame processor ──────────────────────────────────────────────────
    def process_frame(self, raw_frame, frame_idx):
        frame, _ = self.enh.enhance(raw_frame)
        ts = self._ts(frame_idx)
        new_evidence = []

        results = master_model.track(
            frame, persist=True,
            conf=CONF['detection'], iou=NMS_IOU, imgsz=640,
            tracker=TRACKER_NAME, classes=DETECT_CLASSES,
            verbose=False, agnostic_nms=True,
        )

        if not results or results[0].boxes is None or results[0].boxes.id is None:
            return []

        boxes  = results[0].boxes
        bboxes = boxes.xyxy.cpu().numpy()
        confs  = boxes.conf.cpu().numpy()
        clss   = boxes.cls.cpu().numpy().astype(int)
        ids    = boxes.id.cpu().numpy().astype(int)

        person_boxes = [bboxes[i] for i,c in enumerate(clss) if c == PERSON_ID]
        tlight_boxes = [bboxes[i] for i,c in enumerate(clss) if c == TLIGHT_ID]
        vehicle_dets = [(bboxes[i], float(confs[i]), int(clss[i]), int(ids[i]))
                        for i,c in enumerate(clss) if c in VEHICLE_IDS]

        active_ids   = {tid for _,_,_,tid in vehicle_dets}
        signal_state = self.signal.update(frame, frame_idx, tlight_boxes)

        for (bbox, det_conf, cls_id, track_id) in vehicle_dets:
            bbox = clamp_bbox(bbox, self.W, self.H)
            x1, y1, x2, y2 = self.smoother.smooth(track_id, bbox)
            cx = (x1+x2)/2.0; cy = (y1+y2)/2.0
            class_name = master_model.names[cls_id]
            is_moto    = cls_id == COCO['motorcycle']
            exempt     = any(k in class_name.lower() for k in EXEMPT_KW)

            if not in_roi(cx, cy): continue
            if not perspective_ok((x1,y1,x2,y2)): continue

            if track_id not in self._first_seen:
                self._first_seen[track_id] = frame_idx
            self._last_seen[track_id] = frame_idx
            self.lane.update(track_id, cx, cy, frame_idx)

            track_age  = frame_idx - self._first_seen[track_id]
            # V6-V5: MIN_TRACK_AGE_FRAMES now 8 (set in Cell 2)
            old_enough = track_age >= (MIN_TRACK_AGE_FRAMES * PROCESS_EVERY_N)

            # OCR (V6-O1..O9 + V7-O1: vehicle-type-aware crop band)
            detect_and_ocr_plate(frame, (x1,y1,x2,y2), track_id, frame_idx,
                                  anpr=self.anpr, ts=ts, vehicle_class=class_name)
            plate = self.anpr.best_plate(track_id)
            if plate != 'UNKNOWN':
                self._plate_to_track[plate] = track_id

            riders = self._riders((x1,y1,x2,y2), person_boxes)
            frame_area      = self.W * self.H
            bbox_area_ratio = ((x2-x1)*(y2-y1))/frame_area if frame_area else 0

            def check(vtype, rule_conf):
                """V6-V4: each call is fully independent; no shared mutable state."""
                if not old_enough: return
                fused = self._fuse(det_conf, rule_conf)
                if fused < CONF['violation_min']: return
                key = (track_id, vtype)
                activated = self._flag(key, frame_idx, ts, fused,
                                       (x1,y1,x2,y2), track_id,
                                       class_name, riders, exempt, plate)
                if activated:
                    ev = self._events[key]; ev.plate = plate
                    new_evidence.append((ev, frame.copy()))

            # ── 1. Helmet (V6-H*) ──────────────────────────────────────────
            if is_moto and not exempt and bbox_area_ratio >= HELMET_MIN_AREA_RATIO:
                # V6-H1..H6: pass person_boxes so head is extracted from rider bbox
                has_h, hc = has_helmet_v6(frame, (x1,y1,x2,y2), person_boxes)
                if not has_h:
                    check('Helmet non-compliance', hc)

            # ── 2. Triple riding ────────────────────────────────────────────
            if is_moto and riders >= 3 and not exempt:
                check('Triple riding', 0.85)

            # ── 3. Red-light ────────────────────────────────────────────────
            if signal_state == 'red' and not exempt:
                pts = self.lane.tracks.get(track_id, [])
                if len(pts) >= 5:
                    arr  = np.array(pts[-5:])
                    disp = np.hypot(arr[-1,0]-arr[0,0], arr[-1,1]-arr[0,1])
                    if disp > RED_LIGHT_MIN_DISP_PX:
                        check('Red-light violation', 0.80)

            # ── 4. Stop-line ────────────────────────────────────────────────
            if signal_state == 'red' and not exempt:
                if y2 > STOP_LINE_Y:
                    check('Stop-line violation', 0.72)

            # ── 5. Wrong-side ───────────────────────────────────────────────
            if self.lane.is_wrong_side(track_id) and not exempt:
                check('Wrong-side driving', 0.78)

            # ── 6. Illegal parking  (V6-V1 + V6-V2) ────────────────────────
            # V6-V1: dwell raised to 12s
            # V6-V2: explicitly skip if at red light (stationary = waiting, not parked)
            if signal_state != 'red':   # V6-V2
                parking_samples = max(5, int(self.fps * PARKING_SECS / PROCESS_EVERY_N))
                pts = self.lane.tracks.get(track_id, [])
                if len(pts) >= parking_samples:
                    recent = np.array(pts[-parking_samples:])
                    spread_x = float(np.std(recent[:,0]))
                    spread_y = float(np.std(recent[:,1]))
                    stationary = spread_x < 8 and spread_y < 8
                    if stationary and in_forbidden_zone(cx, cy) and not exempt:
                        check('Illegal parking', 0.75)

        self._resolve_stale(active_ids, frame_idx)
        return new_evidence

    def flush(self, frame_idx):
        ts = self._ts(frame_idx)
        for key, ev in list(self._events.items()):
            if ev.state == ViolationState.ACTIVE:
                ev.update(frame_idx, ts, ev.confidence)
                ev.resolve(); self.resolved_events.append(ev)
        self._events.clear()


# Aliases for downstream cells
ViolationEngineV4 = ViolationEngineV6
ViolationEngineV3 = ViolationEngineV6

print('ViolationEngineV6 defined.')
print('  V6-V1 parking 12s | V6-V2 red-light parking exempt')
print('  V6-V5 track-age 8 | V6-V6 ghost 90 frames')


In [ ]:
class EvidenceGenerator:
    def __init__(self, fps, w, h):
        self.fps = fps; self.W = w; self.H = h
        self.buf: deque = deque(maxlen=int(fps * CLIP_SECS))

    def push(self, frame): self.buf.append(frame.copy())

    def save_frame(self, ev: ViolationEvent, frame: np.ndarray) -> str:
        out = frame.copy()
        x1,y1,x2,y2 = ev.bbox
        color = LABEL_COLORS.get(ev.violation, (255,255,0))
        cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
        label = f'{ev.violation} | {ev.plate} | {ev.confidence:.2f}'
        (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        lx = max(0,x1); ly = max(th+12, y1)
        cv2.rectangle(out, (lx,ly-th-10), (lx+tw+6,ly), color, -1)
        cv2.putText(out, label, (lx+3,ly-5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
        cv2.putText(out, f'Track {ev.track_id} | {ev.first_ts} | Signal: {engine.signal.current}',
                    (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
        path = str(FRAMES_DIR / f'{ev.event_id}.jpg')
        cv2.imwrite(path, out, [cv2.IMWRITE_JPEG_QUALITY, 92])
        return path

    def save_clip(self, ev: ViolationEvent) -> str:
        path = str(CLIPS_DIR / f'{ev.event_id}.mp4')
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(path, fourcc, self.fps, (self.W, self.H))
        for f in self.buf: writer.write(f)
        writer.release(); return path


engine   = ViolationEngineV6(FPS, W, H)
evidence = EvidenceGenerator(FPS, W, H)

cap       = cv2.VideoCapture(str(VIDEO_PATH))
frame_idx = 0
pbar      = tqdm(total=TOTAL_FRAMES, desc='Inference v7', unit='fr')
t_start   = time.time()

while cap.isOpened():
    ret, raw = cap.read()
    if not ret: break
    frame_idx += 1
    evidence.push(raw); pbar.update(1)
    if frame_idx % PROCESS_EVERY_N != 0: continue
    newly_active = engine.process_frame(raw, frame_idx)
    for ev, ann_frame in newly_active:
        ev.evidence_img  = evidence.save_frame(ev, ann_frame)
        ev.evidence_clip = evidence.save_clip(ev)

pbar.close(); cap.release()
engine.flush(frame_idx)

elapsed = time.time() - t_start
print(f'Done in {elapsed:.1f}s  ({TOTAL_FRAMES/elapsed:.1f} frames/s effective)')
print(f'  Resolved violations : {len(engine.resolved_events)}')
print(f'  ANPR records        : {len(engine.anpr.records)}')
print(f'  Enhancement stats   : {dict(engine.enh.stats)}')


In [ ]:
events = engine.resolved_events

records_path = REPORTS / 'violations.json'
with open(records_path, 'w') as f:
    json.dump([ev.to_dict() for ev in events], f, indent=2)

anpr_df   = engine.anpr.to_df()
anpr_path = REPORTS / 'anpr_registry.csv'
if not anpr_df.empty: anpr_df.to_csv(anpr_path, index=False)

print(f'violations.json   -> {records_path}  ({len(events)} events)')
print(f'anpr_registry.csv -> {anpr_path}  ({len(engine.anpr.records)} records)')

if not events:
    print('No violations recorded.')
    print('Tips: lower CONF["violation_min"] in Cell 2, or use a longer/denser video.')
else:
    df = pd.DataFrame([ev.to_dict() for ev in events])
    fig, axes = plt.subplots(2, 2, figsize=(16,10))
    fig.suptitle('Traffic Violation Analytics -- v7', fontsize=15, fontweight='bold')

    ax = axes[0,0]
    counts = df['violation'].value_counts()
    colors = [f'#{LABEL_COLORS.get(v,(120,120,200))[2]:02x}{LABEL_COLORS.get(v,(120,120,200))[1]:02x}{LABEL_COLORS.get(v,(120,120,200))[0]:02x}' for v in counts.index]
    bars = ax.barh(counts.index, counts.values, color=colors)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title('Violations by type'); ax.invert_yaxis(); ax.set_xlabel('Count')

    ax = axes[0,1]
    ax.hist(df['confidence'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df['confidence'].mean(), color='red', linestyle='--', label=f'Mean: {df["confidence"].mean():.2f}')
    ax.set_title('Confidence distribution'); ax.legend()
    ax.set_xlabel('Fused confidence'); ax.set_ylabel('Count')

    ax = axes[1,0]
    df['second'] = (df['first_frame'].astype(float)/FPS).astype(int)
    tl = df.groupby('second').size()
    ax.plot(tl.index, tl.values, color='crimson', lw=1.5)
    ax.fill_between(tl.index, tl.values, alpha=0.2, color='crimson')
    ax.set_title('Violation timeline'); ax.set_xlabel('Time (s)'); ax.set_ylabel('Events')

    ax = axes[1,1]
    rpt = df[df['plate']!='UNKNOWN']['plate'].value_counts().head(10)
    if not rpt.empty:
        ax.bar(rpt.index, rpt.values, color='darkorange')
        ax.set_title('Top repeat offenders'); ax.set_xlabel('Plate'); ax.set_ylabel('Events')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5,0.5,'No plates recognised',ha='center',transform=ax.transAxes)
        ax.set_title('Top repeat offenders')

    plt.tight_layout()
    plt.savefig(str(REPORTS/'analytics_v7.png'), dpi=150); plt.show()

    print('\\n' + '='*55)
    print('  SUMMARY -- v7')
    print('='*55)
    print(f'  Resolved events   : {len(df)}')
    print(f'  Unique plates     : {df["plate"].nunique()}')
    print(f'  Mean confidence   : {df["confidence"].mean():.3f}')
    print(f'  Exempt vehicles   : {df["exempt"].sum()}')
    print()
    print('  Breakdown:')
    for vtype, cnt in counts.items():
        print(f'    {vtype:<30} {cnt}')
    print('='*55)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

GT_PATH = BASE / 'gt_labels.json'

if not GT_PATH.exists():
    print('No gt_labels.json found -- skipping metrics.')
    print('Format: [{"frame_idx": 120, "violation": "Helmet non-compliance"}, ...]')
else:
    with open(GT_PATH) as f: gt_raw = json.load(f)
    VTYPES = sorted(set([r['violation'] for r in gt_raw]+[ev.violation for ev in events]))
    gt_map   = {r['frame_idx']: r['violation'] for r in gt_raw}
    pred_map = {ev.first_frame: ev.violation   for ev in events}
    all_f    = sorted(set(list(gt_map)+list(pred_map)))
    y_true = [gt_map.get(f,'No violation')   for f in all_f]
    y_pred = [pred_map.get(f,'No violation') for f in all_f]
    print(classification_report(y_true, y_pred, labels=VTYPES, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=VTYPES)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=VTYPES, yticklabels=VTYPES)
    plt.title('Confusion Matrix -- v7')
    plt.ylabel('Ground Truth'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig(str(REPORTS/'confusion_matrix_v7.png'), dpi=150); plt.show()
    f1s = f1_score(y_true, y_pred, labels=VTYPES, average=None, zero_division=0)
    print(f'  mAP@0.5 proxy (mean F1): {np.mean(f1s):.4f}')


In [ ]:
import zipfile
from google.colab import files
zip_path = '/content/traffic_evidence_v7.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in FRAMES_DIR.glob('*.jpg'): zf.write(f, f'frames/{f.name}')
    for f in CLIPS_DIR.glob('*.mp4'): zf.write(f, f'clips/{f.name}')
    for f in REPORTS.glob('*'): zf.write(f, f'reports/{f.name}')
print(f'Packaged -> {zip_path}')
files.download(zip_path)


## v6 review and change log

### PART 1 - Patch review

| Patch change | Decision | Reason |
|--------------|----------|--------|
| OCR_INTERVAL 50->15 | **MODIFY to 25** | 15 is too aggressive: doubles GPU load with no recall gain beyond 25. 25 = ~1s at 25fps. |
| Plate crop bottom 25% | **KEEP (primary)** | Correct for Indian vehicles; made dual-crop: try 25% first, fallback to 40%. |
| Scale 160//h | **MODIFY** | Integer floor division of a fixed target is inconsistent. Use float scale to target 64px height. |
| CLAHE preprocessing | **KEEP + IMPROVE** | Sound idea; applied on LAB L-channel (better than grayscale CLAHE). |
| paragraph=True | **REMOVE** | paragraph=True merges character boxes into lines, losing per-fragment confidence needed for V6-O6. |
| min_size=10 | **MODIFY to 8** | 10 discards single small characters. 8 is safer minimum. |
| Length-weighted confidence | **KEEP** | Correct: a 7-char fragment is more reliable than a 1-char fragment. |
| Remove OCR double-gate | **PARTIAL** | Remove avg_conf gate (double-gating nullified valid reads). Keep format-validation gate. |
| Person-bbox head extraction | **KEEP + IMPROVE** | Correct principle; added IoU matching and rider-zone constraint to avoid picking pillion rider. |
| Head crop 20% of person | **MODIFY to 30%** | 20% crops too tight and misses tall helmets. 30% is the right range for face+helmet. |
| Edge-density heuristic | **KEEP + IMPROVE** | Valid feature; combined with dark_ratio and saturation for tri-feature scoring. |
| PARKING_SECS 5->30 | **MODIFY to 12** | 30s is impractical (demo videos rarely show 30s of static vehicles). 12s is a real parking dwell. |
| Red-light parking exemption | **KEEP** | Clearly correct: stopped at red is not illegal parking. |
| Per-violation majority voting | **RESTRUCTURE** | Already handled by CONFIRM_FRAMES mechanism; adding a second voting layer introduces unnecessary latency and complexity. V6 keeps CONFIRM_FRAMES=4 and improves the confirm-reset logic. |

### PART 2 - Weaknesses identified

**OCR:** Patch crop may still miss plates on trucks/buses (higher mounting). Scale formula uncapped. No dual-polarity (white-on-dark plates fail on normal threshold). Min-reads not enforced before emitting plate. No retry on near-miss (2 chars off valid format).

**Helmet:** Person-box IoU with full moto bbox picks pillion not rider. Top 20% of person is too tight on most camera angles. Shadow darkens all head regions uniformly causing dark_ratio false positives. No minimum head region size guard before feature extraction.

**Violation engine:** Parking at red light fires as violation before V6-V2 fix. GHOST_FRAMES 60 drops tracks too early at junctions with partial occlusion. track-age gate 5 frames too low for PROCESS_EVERY_N=2 scenarios. Simultaneous triple-riding + helmet was double-confirming same track slots.

### PART 3+4 - Implementation
See Cell 6 (helpers) and Cell 7 (engine) for production code.

### PART 5 — Status (not measured)

The percentage estimates that previously appeared here (e.g. "OCR success
35%→55%", "Helmet recall 58%→72%") were never measured against
`gt_labels.json` or any labeled test set — they were plausibility
estimates presented with false precision. They have been removed.

What can honestly be said about each change without a labeled eval run:

| Change | Status |
|--------|--------|
| Dual-crop, CLAHE, dual-polarity OCR | Plausible improvement, **unvalidated** |
| Tri-feature helmet heuristic | Plausible improvement, **unvalidated** — still a hand-tuned heuristic, not a trained classifier; known to be vulnerable to turbans, very curly dark hair, and unusual lighting it wasn't tuned against |
| 12s parking dwell + red-light exemption | Logically sound fix for a real false-positive source |
| Ghost-90 / track-age-8 / bbox cooldown | Logically sound, reduces duplicate-event risk |

To get real numbers, run Cell 10 (evaluation) against a `gt_labels.json`
built from manually annotated frames of your actual test video, and
report the classification_report / mAP output it produces — not an
estimate.

---

## v7 — correcting a wrong root-cause claim, fixing the real one

A review claimed the engine has an early-exit architecture: "find first
violation → stop evaluating," and used a screenshot showing
`Illegal Parking [UNKNOWN]` next to a clearly visible plate and a bare
head as evidence.

**That claim was checked against the actual code in `ViolationEngineV6.process_frame()`
and is false.** The six violation checks (helmet, triple-riding,
red-light, stop-line, wrong-side, parking) are six independent sibling
`if` blocks per vehicle per frame. None of them contains `continue`,
`break`, or an early `return` that would skip the others. Every vehicle
is evaluated against all six rules, every processed frame. Each
violation type gets its own `ViolationEvent` keyed by
`(track_id, violation_type)`, so a helmet violation and a parking
violation on the same vehicle would appear as two separate events with
two separate evidence images — not merged or overwritten.

**The screenshot symptom was real, and the actual causes were:**

| ID | Root cause | Fix |
|----|-----------|-----|
| V7-V1 | `ev.plate` was set once at `activate()` and never updated again. OCR is throttled (`OCR_INTERVAL` frames per track), so if a violation activates before that track's first successful OCR read lands, the event is permanently stamped `UNKNOWN` even though OCR succeeds on the same track moments later. | `_flag()` now refreshes `ev.plate` on every `update()` call if a better (non-UNKNOWN) read becomes available, without overwriting a good read with a worse one. |
| V7-O1 | OCR crop was a single fixed bottom-25%/40% band for every vehicle type. A bus/truck bbox is much taller, so that band lands on the wheel well or road surface, not the plate. Motorcycle plates also sit higher relative to the bbox bottom than car plates. | `detect_and_ocr_plate()` now takes `vehicle_class` and selects a crop band per type (motorcycle / bus / truck / car / bicycle) instead of one band for all vehicles. |
| — | "No Helmet" not appearing for a parked vehicle is not a bug if that vehicle is a car/truck — the helmet check is correctly gated on `is_moto` and never runs for four-wheelers. | No fix needed; documented so it isn't mistaken for a defect again. |

No change was made to the per-vehicle evaluation loop itself, because
there was nothing wrong with it.

